In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [1]:
!pip install -q ultralytics

In [ ]:
from pathlib import Path
from datetime import datetime

import json
import platform
import random
import shutil
import yaml

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics

from IPython.display import display
from PIL import Image
from ultralytics import YOLO
from concurrent.futures import ThreadPoolExecutor

In [ ]:
DRIVE_DATASET_ROOT = Path("/content/drive/MyDrive/Yolo_road2022_dataset/RDD2022_China_MotorBike")
LOCAL_DATASET_ROOT = Path("/content/RDD2022_China_MotorBike_1")


def copy_file(src_file, src_root, dst_root):
    relative_path = src_file.relative_to(src_root)
    dest_file = dst_root / relative_path
    dest_file.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src_file, dest_file)


for split in ["train", "val"]:
    src_dir = DRIVE_DATASET_ROOT / split
    dst_dir = LOCAL_DATASET_ROOT / split
    files = [p for p in src_dir.rglob("*") if p.is_file()]
    with ThreadPoolExecutor(max_workers=16) as executor:
        futures = [
            executor.submit(copy_file, f, src_dir, dst_dir) for f in files
        ]
        for future in futures:
            future.result()

Parallel copy completed successfully!


In [6]:
DATASET_ROOT = LOCAL_DATASET_ROOT

TRAIN_IMAGES_DIR = DATASET_ROOT / "train/images"
TRAIN_LABELS_DIR = DATASET_ROOT / "train/labels"

VAL_IMAGES_DIR = DATASET_ROOT / "val/images"
VAL_LABELS_DIR = DATASET_ROOT / "val/labels"

CONFIGS_DIR = DATASET_ROOT / "configs"
DATA_YAML_PATH = CONFIGS_DIR / "rdd2022_china_motorbike.yaml"

# Drive
REPORTS_DIR = DRIVE_DATASET_ROOT  / "reports"
RUNS_ROOT = DRIVE_DATASET_ROOT / "runs"

SPLIT_REPORT_PATH = REPORTS_DIR / "rdd2022_china_motorbike_split_manifest.json"

print("Data YAML :", DATA_YAML_PATH.resolve())
print("Runs root :", RUNS_ROOT.resolve())

Data YAML : /content/RDD2022_China_MotorBike_1/configs/rdd2022_china_motorbike.yaml
Runs root : /content/drive/MyDrive/Yolo_road2022_dataset/RDD2022_China_MotorBike/runs


In [ ]:
CLASS_NAMES = {
    0: "D00",
    1: "D10",
    2: "D20",
    3: "D40",
}

CONFIGS_DIR.mkdir(parents=True,exist_ok=True)
dataset_configuration = {
    "path": str(LOCAL_DATASET_ROOT.resolve()),
    "train": "train/images",
    "val": "val/images",
    "names": CLASS_NAMES,
}

with DATA_YAML_PATH.open("w", encoding="utf-8") as yaml_file:
    yaml.safe_dump(
        dataset_configuration,
        yaml_file,
        sort_keys=False,
        allow_unicode=True,
    )

print("Dataset YAML:", DATA_YAML_PATH.resolve())
print(DATA_YAML_PATH.read_text(encoding="utf-8"))

Dataset YAML: /content/RDD2022_China_MotorBike_1/configs/rdd2022_china_motorbike.yaml
path: /content/RDD2022_China_MotorBike_1
train: train/images
val: val/images
names:
  0: D00
  1: D10
  2: D20
  3: D40



In [8]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp"}

train_images = sorted(
    path
    for path in TRAIN_IMAGES_DIR.iterdir()
    if path.suffix.lower() in IMAGE_SUFFIXES
)

val_images = sorted(
    path
    for path in VAL_IMAGES_DIR.iterdir()
    if path.suffix.lower() in IMAGE_SUFFIXES
)

train_labels = sorted(TRAIN_LABELS_DIR.glob("*.txt"))
val_labels = sorted(VAL_LABELS_DIR.glob("*.txt"))

print("Training images  :", len(train_images))
print("Training labels  :", len(train_labels))
print("Validation images:", len(val_images))
print("Validation labels:", len(val_labels))

Training images  : 1547
Training labels  : 1547
Validation images: 387
Validation labels: 387


**Training configuration**


In [9]:
MODEL_ID = "yolo26n.pt"
IMAGE_SIZE = 640
EPOCHS = 50
PATIENCE = 15
TRAIN_BATCH = 15
VALIDATION_BATCH = 8
SEED = 42
WORKERS = 0
RUN_NAME = "yolo26n_rdd2022_baseline"
DEVICE = 0 if torch.cuda.is_available() else "cpu"

In [10]:
training_configuration = {
    "model" : MODEL_ID,
    "data" : str(DATA_YAML_PATH.resolve()),
    "epochs" : EPOCHS,
    "imgsz" : IMAGE_SIZE,
    "batch" : TRAIN_BATCH,
    "device" : DEVICE,
    "workers" : WORKERS,
    "patience" : PATIENCE,
    "optimizer" : "auto",
    "warmup_epochs" : 1.0,
    "close_mosaic" : 10,
    "amp" : True,
    "cache" : False,
    "seed" : SEED,
    "deterministic" : True,
    "val" : True,
    "plots" : True,
    "save" : True,
    "save_period" : 5,
    "project" : str(RUNS_ROOT.resolve()),
    "name" : RUN_NAME,
    "exist_ok" : False,
    "verbose" : True
}

for key, value in training_configuration.items():
    print(f"{key:<18}: {value}")

model             : yolo26n.pt
data              : /content/RDD2022_China_MotorBike_1/configs/rdd2022_china_motorbike.yaml
epochs            : 50
imgsz             : 640
batch             : 15
device            : 0
workers           : 0
patience          : 15
optimizer         : auto
warmup_epochs     : 1.0
close_mosaic      : 10
amp               : True
cache             : False
seed              : 42
deterministic     : True
val               : True
plots             : True
save              : True
save_period       : 5
project           : /content/drive/MyDrive/Yolo_road2022_dataset/RDD2022_China_MotorBike/runs
name              : yolo26n_rdd2022_baseline
exist_ok          : False
verbose           : True


**Save training configuration**


In [11]:
training_config_path = (REPORTS_DIR / "rdd2022_yolo26n_training_config.json")

serializable_configuration =  {
    key: (str(value) if isinstance(value, Path) else value)
    for key, value in training_configuration.items()
}

training_config_path.write_text(
    json.dumps(serializable_configuration, indent=2), encoding="utf-8"
)

print("Saved:", training_config_path.resolve())

Saved: /content/drive/MyDrive/Yolo_road2022_dataset/RDD2022_China_MotorBike/reports/rdd2022_yolo26n_training_config.json


**Full fine-tuning**


In [ ]:
RESUME_FROM = None

if RESUME_FROM is None:
    model = YOLO(MODEL_ID)
    model.info()
    
    training_results = model.train(**training_configuration)

else:
    RESUME_FROM = Path(RESUME_FROM)
    
    if not RESUME_FROM.is_file():
        raise FileNotFoundError(RESUME_FROM)

    model = YOLO(str(RESUME_FROM))
    training_results = model.train(resume=True)

YOLO26n summary: 260 layers, 2,572,280 parameters, 0 gradients, 6.2 GFLOPs
Ultralytics 8.4.137 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=15, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/RDD2022_China_MotorBike_1/configs/rdd2022_china_motorbike.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=

**Locate training output**


In [ ]:
RUN_DIRECTORY = Path(model.trainer.save_dir)
BEST_MODEL_PATH = RUN_DIRECTORY / "weights" / "best.pt"
LAST_MODEL_PATH = RUN_DIRECTORY / "weights" / "last.pt"
RESULTS_CSV_PATH = RUN_DIRECTORY / "results.csv"

print("Run directory:", RUN_DIRECTORY.resolve())
print("Best model   :", BEST_MODEL_PATH.resolve())
print("Last model   :", LAST_MODEL_PATH.resolve())

**Inspect training history**


In [ ]:
training_history = pd.read_csv(RESULTS_CSV_PATH)

training_history.columns = [
    column.strip() for column in training_history.columns
]

print("Completed epochs:", len(training_history))
display(training_history.tail())

In [ ]:
for column in training_history.columns:
    print(column)

**Find highest recorded validation mAP**


In [ ]:
map_50_95_column = next(
    (
        column for column in training_history.columns
        if "metrics/mAP50-95" in column
    ),
    None
)

map_50_column = next(
    (
        column for column in training_history.columns
        if ("metrics/mAP50" in column and "50-95" not in column)
    ),
    None
)

if map_50_95_column is not None:
    highest_map_row_index = training_history[map_50_95_column].idxmax()
    
    highest_map_row = training_history.loc[highest_map_row_index]

    print("Highest recorded mAP50-95 epoch:", int(highest_map_row["epoch"]) + 1)
    
    print("Highest recorded mAP50-95:", round(float(highest_map_row[map_50_95_column]), 4))

    if map_50_column is not None:
        print("mAP50 at that epoch:", round(float(highest_map_row[map_50_column]), 4))

**Display generated training artifacts**


In [ ]:
artifact_names = [
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "F1_curve.png",
    "val_batch0_labels.jpg",
    "val_batch0_pred.jpg",
]

for artifact_name in artifact_names:
    artifact_path = RUN_DIRECTORY / artifact_name

    if artifact_path.is_file():
        print("\n", artifact_name)
        display(Image.open(artifact_path))

**Independent final validation**


In [ ]:
best_model = YOLO(str(BEST_MODEL_PATH))

for class_id, class_name in (best_model.names.items()):
    print(class_id, "→", class_name)

In [ ]:
validation_metrics = best_model.val(
    data=str(DATA_YAML_PATH.reslove()),
    split="val",
    imgsz=IMAGE_SIZE,
    batch=VALIDATION_BATCH,
    device=DEVICE,
    workers=WORKERS,
    conf=0.001,
    plots=True,
    project=str(RUN_DIRECTORY.reslove()),
    name="final_validation",
    exist_ok=True,
    verbose=True
)

In [ ]:
mean_precision = getattr(validation_metrics.box, "mp", None)
mean_recall = getattr(validation_metrics.box, "mr", None)

map_50 = float(validation_metrics.box.map50)
map_75 = float(validation_metrics.box.map75)
map_50_95 = float(validation_metrics.box.map)


if mean_precision is not None:
    print("Mean Precision :", round(float(mean_precision), 4))

if mean_recall is not None:
    print("Mean Recall    :", round(float(mean_recall), 4))

print("mAP@0.50       :", round(map_50, 4))
print("mAP@0.75       :", round(map_75, 4))
print("mAP@0.50:0.95  :", round(map_50_95, 4))

**Per-class mAP**


In [ ]:
per_class_maps = np.asarray(validation_metrics.box.maps,
dtype=np.float32)

print(f"{'Class':<30}" f"{'mAP50-95':>12}")
print("-" * 42)

for class_id, class_map in enumerate(per_class_maps):
    class_name = best_model.names[class_id]
    print(f"{class_name:<30}" f"{float(class_map):>12.4f}")

In [ ]:
class_names = [
    best_model.names[class_id]
    for class_id in range(len(per_class_maps))
]

plt.figure(figsize=(10, 5))
bars = plt.bar(
    class_names,
    per_class_maps,
    color=[
        "royalblue",
        "darkorange",
        "seagreen",
        "crimson",
    ],
)

plt.ylim(0, max(1.0, float(per_class_maps.max()) * 1.15))
plt.ylabel("mAP@0.50:0.95")
plt.title("RDD2022 per-class validation performance")
plt.xticks(rotation=15)
plt.grid(axis="y", alpha=0.25)

for bar, value in zip(bars, per_class_maps):
    plt.text(
        bar.get_x()
        + bar.get_width() / 2,
        bar.get_height(),
        f"{float(value):.3f}",
        ha="center",
        va="bottom",
    )

plt.tight_layout()
plt.show()